In [39]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

import numpy as np
import pandas as pd

In [40]:
%run 00.ipynb

Export Folder : standardized_exports
Results Folder: results
24
Datasets Found: 22
agri_medium
agri_small
energy_medium
energy_small
favorita-train_medium
favorita-train_medium_cov
favorita-train_small
favorita-train_small_cov
favorita-transaction_medium
favorita-transaction_small
m4-daily_medium
m4-daily_small
m4-hourly_medium
m4-hourly_small
m4-quarterly_medium
m4-quarterly_small
m4-weekly_medium
m4-weekly_small
m4-yearly_medium
m4-yearly_small
traffic_medium
traffic_small
5
agri_medium               rows=15343      series=246      horizon=5
5
agri_small                rows=14428      series=246      horizon=5
24
energy_medium             rows=177600     series=370      horizon=24
24
energy_small              rows=44400      series=370      horizon=24
28
favorita-train_medium     rows=1985155    series=4504     horizon=28
favorita-train_medium_cov -> ERROR: No horizon configured for favorita-train_medium_cov
28
favorita-train_small      rows=609961     series=4504     horizon=28
favo

In [41]:
RESULT_FILE = "classical_results.csv"

initialize_results_file(
    RESULT_FILE
)

completed_runs = get_completed_runs(
    RESULT_FILE
)

print(
    f"Completed Runs: {len(completed_runs)}"
)

Completed Runs: 0


In [42]:
def naive_forecast(
    train_series,
    horizon
):

    last_value = train_series.iloc[-1]

    return np.repeat(
        last_value,
        horizon
    )


In [43]:
def seasonal_naive_forecast(
    train_series,
    horizon,
    season_length
):

    if len(train_series) < season_length:

        return naive_forecast(
            train_series,
            horizon
        )

    season = train_series.iloc[
        -season_length:
    ].values

    repeats = int(
        np.ceil(
            horizon / season_length
        )
    )

    forecast = np.tile(
        season,
        repeats
    )

    return forecast[:horizon]

In [44]:
def arima_forecast(
    train_series,
    horizon
):

    try:

        model = ARIMA(
            train_series,
            order=(1,1,1)
        )

        fitted = model.fit()

        forecast = fitted.forecast(
            steps=horizon
        )

        return np.asarray(
            forecast
        )

    except:

        return naive_forecast(
            train_series,
            horizon
        )

In [45]:
def sarima_forecast(
    train_series,
    horizon,
    season_length
):

    try:

        model = SARIMAX(
            train_series,
            order=(1,1,1),

            seasonal_order=(
                1,
                0,
                1,
                season_length
            ),

            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(
            disp=False
        )

        forecast = fitted.forecast(
            horizon
        )

        return np.asarray(
            forecast
        )

    except:

        return naive_forecast(
            train_series,
            horizon
        )

In [46]:
def get_season_length(
    dataset_name
):

    if "energy" in dataset_name:
        return 24

    if "electricity" in dataset_name:
        return 24

    if "metr_la" in dataset_name:
        return 12

    if "favorita" in dataset_name:
        return 7

    if "m4" in dataset_name:
        return 12

    return 1

In [47]:
MAX_SERIES_ARIMA = 100

def evaluate_model(
    train_df,
    test_df,
    horizon,
    forecast_function,
    **kwargs
):

    actuals = []
    forecasts = []

    # ---------------------------------
    # Limit ARIMA/SARIMA on huge datasets
    # ---------------------------------

    series_ids = train_df["unique_id"].unique()

    if forecast_function in [
        arima_forecast,
        sarima_forecast
    ]:
        series_ids = series_ids[:MAX_SERIES_ARIMA]

    # ---------------------------------

    for uid in series_ids:

        train_group = train_df[
            train_df["unique_id"] == uid
        ]

        test_group = test_df[
            test_df["unique_id"] == uid
        ]

        if len(test_group) == 0:
            continue

        train_values = train_group["y"]

        try:

            pred = forecast_function(
                train_values,
                horizon,
                **kwargs
            )

        except:

            pred = naive_forecast(
                train_values,
                horizon
            )

        actuals.extend(
            test_group["y"].values
        )

        forecasts.extend(pred)

    metrics = calculate_metrics(
        actuals,
        forecasts
    )

    return metrics

In [48]:
MODELS = {

    "Naive": {
        "func": naive_forecast,
        "kwargs": {}
    },

    "SeasonalNaive": {
        "func": seasonal_naive_forecast,
        "kwargs": None
    },

    "ARIMA": {
        "func": arima_forecast,
        "kwargs": {}
    },

    "SARIMA": {
        "func": sarima_forecast,
        "kwargs": None
    }
}

In [49]:
for dataset_name in AVAILABLE_DATASETS:

    print("\n" + "=" * 80)
    print(dataset_name)

    try:

        df, train_df, test_df, horizon = (
            prepare_dataset(
                dataset_name
            )
        )

        season_length = (
            get_season_length(
                dataset_name
            )
        )

        for model_name, config in MODELS.items():

            if (
                dataset_name,
                model_name
            ) in completed_runs:

                print(
                    f"SKIP : {model_name}"
                )
                continue

            print(
                f"RUN  : {model_name}"
            )

            start = start_timer()

            kwargs = config["kwargs"]

            if model_name in [
                "SeasonalNaive",
                "SARIMA"
            ]:

                metrics = evaluate_model(
                    train_df,
                    test_df,
                    horizon,
                    config["func"],
                    season_length=season_length
                )

            else:

                metrics = evaluate_model(
                    train_df,
                    test_df,
                    horizon,
                    config["func"]
                )

            runtime = stop_timer(
                start
            )

            result = {

                "model":
                model_name,

                "dataset":
                dataset_name,

                "n_series":
                df["unique_id"].nunique(),

                "n_rows":
                len(df),

                "horizon":
                horizon,

                "runtime_seconds":
                runtime,

                **metrics
            }

            append_result(
                RESULT_FILE,
                result
            )

            print(
                f"MAE={metrics['MAE']:.4f}"
            )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )


agri_medium
5
RUN  : Naive
MAE=590406.6449
RUN  : SeasonalNaive
MAE=590406.6449
RUN  : ARIMA
MAE=214449.3395
RUN  : SARIMA
MAE=204309.5870

agri_small
5
RUN  : Naive
MAE=590406.6449
RUN  : SeasonalNaive
MAE=590406.6449
RUN  : ARIMA
MAE=209383.7424
RUN  : SARIMA
MAE=204309.5870

energy_medium
24
RUN  : Naive
MAE=216.5903
RUN  : SeasonalNaive
MAE=223.3347
RUN  : ARIMA
MAE=38.5519
RUN  : SARIMA
MAE=36.3310

energy_small
24
RUN  : Naive
MAE=216.5903
RUN  : SeasonalNaive
MAE=223.3347
RUN  : ARIMA
MAE=40.4887
RUN  : SARIMA
MAE=464617782362984808599497093666766431862876351168917323316017838412513140175792834369953297302199560355409035264.0000

favorita-train_medium
28
RUN  : Naive
MAE=4.0295
RUN  : SeasonalNaive
MAE=3.9223
RUN  : ARIMA
MAE=2.1229
RUN  : SARIMA
MAE=2.1130

favorita-train_medium_cov
ERROR: No horizon configured for favorita-train_medium_cov

favorita-train_small
28
RUN  : Naive
MAE=4.0295
RUN  : SeasonalNaive
MAE=3.9223
RUN  : ARIMA
MAE=2.0906
RUN  : SARIMA
MAE=2.0907

favorit

In [50]:
results_df = pd.read_csv(
    RESULTS_FOLDER /
    RESULT_FILE
)

results_df.sort_values(
    ["dataset", "model"]
)

,model,dataset,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE
2,ARIMA,agri_medium,246,15343,5,13.942256,214449.339510,6.649463e+05,9.278785
0,Naive,agri_medium,246,15343,5,1.266055,590406.644935,3.723966e+06,10.800519
3,SARIMA,agri_medium,246,15343,5,0.502644,204309.587000,6.231367e+05,9.294803
1,SeasonalNaive,agri_medium,246,15343,5,1.435681,590406.644935,3.723966e+06,10.800519
6,ARIMA,agri_small,246,14428,5,14.784277,209383.742422,6.600859e+05,9.088212
...,...,...,...,...,...,...,...,...,...
73,SeasonalNaive,traffic_medium,207,99360,48,0.679582,4.381174,9.954850e+00,8.769702
78,ARIMA,traffic_small,207,24840,48,15.701206,6.082160,1.175885e+01,12.483788
76,Naive,traffic_small,207,24840,48,0.668775,4.381174,9.954850e+00,8.769702
79,SARIMA,traffic_small,207,24840,48,0.251096,4.386917,1.071836e+01,9.469298
